In [1]:
import sqlite3
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import math

In [17]:
file='starrail_weekly.db'
conn=sqlite3.connect(file)
conn.execute("PRAGMA foreign_keys = ON;")
cur=conn.cursor()

In [ ]:
API_KEY = "API KEY HERE"
youtube = build("youtube","v3",developerKey=API_KEY)

In [4]:
#SEARCH/GET VIDEO IDS
def search_videos(query, total_results_needed):
    videos = []
    next_page_token = None

    while len(videos) < total_results_needed:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=50,  #
            pageToken=next_page_token  
        ).execute()

        for item in response.get("items", []):
            videos.append(item["id"]["videoId"])

        next_page_token = response.get("nextPageToken")
        
        if not next_page_token:
            break

    return videos[:total_results_needed]

In [5]:
def get_videos(video_ids,search_query):
    response = youtube.videos().list(
        part="snippet,statistics",
        id=video_ids
    ).execute()

    if not response["items"]:
        return

    item = response["items"][0]

    snippet = item["snippet"]
    stats = item.get("statistics",{})

    cur.execute("""INSERT OR IGNORE INTO video
        VALUES (?,?,?,?,?,?,?,?,?)
        """,
        (
            video_ids,
            snippet["title"],
            snippet["channelId"],
            snippet["channelTitle"],
            snippet["publishedAt"],
            int(stats.get("viewCount",0)),
            int(stats.get("likeCount",0)),
            int(stats.get("commentCount",0)),
            search_query

        ))
    return {"video_id":video_ids,
            "channel_id":snippet["channelId"]}

In [6]:

def get_comments(video_id, max_comments, channel_id):

    comments = []
    next_page = None

    try:
        while len(comments) < max_comments:

            response = youtube.commentThreads().list(
                part="snippet,replies",
                videoId=video_id,
                maxResults=100,
                pageToken=next_page,
                textFormat="plainText",
                order="relevance"
            ).execute()

            for item in response["items"]:

                comment = item["snippet"]["topLevelComment"]
                data = comment["snippet"]
                comment_channel_id = data.get("authorChannelId",{}).get("value")

                
                if channel_id and comment_channel_id == channel_id:
                    continue

                comments.append({
                    "comment_id": comment["id"],
                    "video_id": video_id,
                    "parent_id": None,
                    "author": data["authorDisplayName"],
                    "comment_body": data["textDisplay"],
                    "comment_score": data["likeCount"],
                    "reply_count": item["snippet"]["totalReplyCount"],
                    "published_at": data["publishedAt"]
                })
                # Replies
                if "replies" in item:
                    for reply in item["replies"]["comments"]:

                        if len(comments) >= max_comments:
                            break

                        reply_data = reply["snippet"]

                        comments.append({
                            "comment_id": reply["id"],
                            "video_id": video_id,
                            "parent_id": comment["id"],
                            "author": reply_data["authorDisplayName"],
                            "comment_body": reply_data["textDisplay"],
                            "comment_score": reply_data["likeCount"],
                            "reply_count": 0,
                            "published_at": reply_data["publishedAt"]
                        })

                if len(comments) >= max_comments:
                    break

            next_page = response.get("nextPageToken")

            if not next_page:
                break

    except HttpError as e:
        print(f"Error Scraping {video_id}: {e}")

    return comments

In [7]:
def insert_comment(comment):
    cur.execute("""
    INSERT OR IGNORE INTO video_comments
    VALUES (?,?,?,?,?,?,?,?)
    """,

    (
        comment["comment_id"],
        comment["video_id"],
        comment["parent_id"],
        comment["author"],
        comment["comment_body"],
        comment["comment_score"],
        comment["reply_count"],
        comment["published_at"],
    ))

In [8]:
###SAMPLE TERMS

# HONKAI STAR RAIL 4.X TRAILER/TEASER/LIVESTREAM/SPECIALPROGRAM
# HONKAI STAR RAIL 4.X UPDATE/REVIEW/ANALYSIS
# HONKAI STAR RAIL 4.X NEW CHARACTER/BANNER/PULL/REVIEW
# HONKAI STAR RAIL 4.X AMAZING/HYPE/DISAPPOINTING/BAD

search_term = ["Honkai Star Rail 4.5",
               "Honkai Star Rail 4.5 Livestream",
               "Honkai Star Rail 4.5 Review",
               "Honkai Star Rail 4.5 Robin Summeretto",
               "Honkai Star Rail 4.5 Aventurine Waveflair"
               ]

video_ids=[]

for term in search_term:
    video_ids.extend(search_videos(search_term,200))

In [9]:
len(set(video_ids))

84

In [ ]:
num=1
n=0
for vid in set(video_ids):
    video = get_videos(vid,1)

    if video is None:
        continue
    comments = get_comments(
        video_id = video["video_id"],
        max_comments=500,
        channel_id=video["channel_id"])

    print("Scraping: ",video["video_id"]," : ",len(comments)," : ",num)
    num +=1
    n+=1
    for comment in comments:
        insert_comment(comment)
    

Scraping:  721TPdJpE4g  :  21  :  1
Scraping:  x2_65RHsVqw  :  14  :  2
Scraping:  nMh5b2ygQtk  :  74  :  3
Scraping:  A6NeKMy3uiI  :  500  :  4
Scraping:  f6a_slIMY2o  :  33  :  5
Scraping:  Nu3cilWQdlY  :  12  :  6
Scraping:  BdaO6tGkXYg  :  46  :  7
Scraping:  VWLHobHoty4  :  4  :  8
Scraping:  tjgYvHFScEA  :  59  :  9
Scraping:  p-Ji7mKNCbQ  :  126  :  10
Scraping:  t2Hae1hTo8Y  :  0  :  11
Scraping:  jKgFWyETh6E  :  500  :  12
Scraping:  0EL6UAm4dD8  :  360  :  13
Scraping:  oZKpjCIO53Y  :  39  :  14
Scraping:  kASJyxNetb4  :  73  :  15
Scraping:  _IVF5Wk7cvo  :  3  :  16
Scraping:  GBbwnIl8Cbs  :  15  :  17
Scraping:  VWwstPQchB4  :  5  :  18
Scraping:  bvhVn9SPbjA  :  12  :  19
Scraping:  iTvksd4F7aI  :  49  :  20
Scraping:  ux12Vt8jtao  :  10  :  21
Scraping:  6gniMFVO9Ho  :  4  :  22
Scraping:  C7dsZy2WH2E  :  103  :  23
Scraping:  BHA-JJrA-uc  :  33  :  24
Scraping:  4bJEFb2AtUM  :  107  :  25
Scraping:  M32brl3CLUA  :  159  :  26
Scraping:  YbSWUi4arsk  :  6  :  27
Scraping:

In [20]:
conn.commit()
conn.close()

In [ ]:
### Table schema
cur.executescript("""
CREATE TABLE IF NOT EXISTS video (
      video_id TEXT PRIMARY KEY,
      title TEXT,
      channel_id TEXT,            
      channel_name TEXT,       
      video_published_at DATETIME,     
      view_count INTEGER,
      like_count INTEGER,
      comment_count INTEGER,
      search_query TEXT           
);

CREATE TABLE IF NOT EXISTS video_comments (
      comment_id TEXT PRIMARY KEY,
      video_id TEXT NOT NULL,
      parent_id TEXT NULL,
      author TEXT,
      comment_body TEXT,
      comment_score INTEGER,
      reply_count INTEGER,
      published_at DATETIME,
      FOREIGN KEY (video_id) REFERENCES video(video_id)

);
""")